# 08 — Risk Engine

**Objective**: run `src/services/delivery_metrics.py`, `src/services/financial_metrics.py`, and `src/services/risk_engine.py` end to end across the full 7-project portfolio, producing the final combined RAG status per project — and compare the result against Phase 1's independently hand-computed sample report (`docs/sample_expected_elt_report.md`), which predicted this exact outcome before any of this code existed.

**Dependencies**: notebooks 05-07 (unified projects + metrics); `config/risk_rules.yaml`.

**Configuration**: `AS_OF = 2026-09-15`, consistent with notebooks 05-07.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier, risk_engine

jira_client = build_default_jira_client()
fin_source = CSVFinancialDataSource()
mapping = project_unifier.load_project_mapping()
AS_OF = date(2026, 9, 15)

portfolio = project_unifier.build_unified_portfolio(mapping, jira_client, fin_source, AS_OF)

## Assessing every project

In [2]:
def assess(project):
    entry = mapping.entries[project.project_id]

    issues, latest_sprint = None, None
    if entry.jira_key:
        issues = jira_client.get_project_issues(entry.jira_key, as_of=AS_OF).records
        prev = jira_client.get_previous_sprints(entry.jira_key, 1, as_of=AS_OF)
        latest_sprint = prev[0] if prev else None

    financial_record = None
    if entry.finance_project_id:
        latest_period = fin_source.get_latest_reporting_period(entry.finance_project_id)
        raw = fin_source.get_project_finances(entry.finance_project_id, latest_period) if latest_period else None
        financial_record = raw.with_calculated_fields() if raw else None

    return risk_engine.assess_project_risk(project, issues, latest_sprint, financial_record)

assessments = {p.project_id: assess(p) for p in portfolio}

rows = []
for p in portfolio:
    a = assessments[p.project_id]
    rows.append({
        "project": p.project_key,
        "delivery_risk": a.delivery_risk.severity.value if a.delivery_risk else "N/A",
        "financial_risk": a.financial_risk.severity.value if a.financial_risk else "N/A",
        "combined_rag": a.combined_rag.value,
        "reason_codes": ", ".join(a.reason_codes),
    })
pd.DataFrame(rows)

,project,delivery_risk,financial_risk,combined_rag,reason_codes
0,PHX,HIGH,HIGH,RED,"AGED_BLOCKER, DEPENDENCY_RISK, FORECAST_OVERRU..."
1,ORCA,HIGH,HIGH,RED,"AGED_BLOCKER, DEPENDENCY_RISK, HIGH_BUDGET_CON..."
2,NOVA,HIGH,HIGH,RED,"AGED_BLOCKER, DEPENDENCY_RISK, FORECAST_OVERRU..."
3,TITAN,HIGH,HIGH,RED,"AGED_BLOCKER, DEPENDENCY_RISK, FORECAST_OVERRU..."
4,LYNX,HIGH,HIGH,RED,"AGED_BLOCKER, DEPENDENCY_RISK, FORECAST_OVERRU..."
5,QSR,MEDIUM,N/A,AMBER,LOW_SPRINT_COMPLETION
6,FIN-10007,N/A,LOW,AMBER,


## Cross-check against Phase 1's prediction

`docs/sample_expected_elt_report.md` predicted this exact RAG distribution by hand, before `delivery_metrics.py`, `financial_metrics.py`, or `risk_engine.py` existed: all 5 fully-mapped projects RED, the Jira-only project capped at AMBER by its missing finance mapping. This is the first time that prediction has been checked against real, tested code.

In [3]:
from src.models.common import RAGStatus

expected_red = {"PROJECT-10001", "PROJECT-10002", "PROJECT-10003", "PROJECT-10004", "PROJECT-10005"}
actual_red = {pid for pid, a in assessments.items() if a.combined_rag == RAGStatus.RED}

print(f"Phase 1 predicted RED: {sorted(expected_red)}")
print(f"Phase 6 computed RED:  {sorted(actual_red)}")
assert actual_red == expected_red, "Phase 6's risk engine disagrees with Phase 1's hand-computed prediction"
print("\nMatch confirmed.")

Phase 1 predicted RED: ['PROJECT-10001', 'PROJECT-10002', 'PROJECT-10003', 'PROJECT-10004', 'PROJECT-10005']
Phase 6 computed RED:  ['PROJECT-10001', 'PROJECT-10002', 'PROJECT-10003', 'PROJECT-10004', 'PROJECT-10005']

Match confirmed.


## Full evidence trail for one project (PHX)

Every HIGH-severity finding carries concrete evidence (Accuracy Check 7) — never a bare "this project seems risky".

In [4]:
phx_assessment = assessments["PROJECT-10001"]

print(f"DELIVERY RISK: {phx_assessment.delivery_risk.severity.value}")
print(f"  {phx_assessment.delivery_risk.description}")
for line in phx_assessment.delivery_risk.evidence:
    print(f"    - {line}")

print(f"\nFINANCIAL RISK: {phx_assessment.financial_risk.severity.value}")
print(f"  {phx_assessment.financial_risk.description}")
for line in phx_assessment.financial_risk.evidence:
    print(f"    - {line}")

print(f"\nCOMBINED: {phx_assessment.combined_rag.value}")
print(f"reason_codes: {phx_assessment.reason_codes}")

DELIVERY RISK: HIGH
  2 delivery risk condition(s) triggered: AGED_BLOCKER, DEPENDENCY_RISK
    - PHX-17: blocked 240 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-4: blocked 235 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-15: blocked 224 days (Unresolved dependency: PHX-3)
    - PHX-26: blocked 224 days (Flagged blocked via blocker_status field (source provided no reason text))
    - PHX-6 depends on PHX-4: not done (status=IN_REVIEW)
    - PHX-14 depends on PHX-2: not done (status=IN_PROGRESS)
    - PHX-22 depends on PHX-9: not done (status=IN_REVIEW)

FINANCIAL RISK: HIGH
  4 financial risk condition(s) triggered: FORECAST_OVERRUN, HIGH_BUDGET_CONSUMPTION, NEGATIVE_REMAINING_BUDGET, SPEND_AHEAD_OF_PROGRESS
    - approved_budget=60,533.77
    - forecast_spend=68,847.48
    - forecast_variance=-8,313.71
    - budget_consumption_pct=111.9%
    - actual_spend=67,749.24
    - approved_budget=60,53

## The two mapping-gap projects: capped at AMBER, never GREEN

This is `combined_risk_matrix`'s `unknown_default` rule in action: missing information about half a project's risk profile is never grounds for calling it healthy, even when the side that IS visible looks fine (Helios's financial risk is LOW).

In [5]:
for pid in ["PROJECT-10006", "PROJECT-10007"]:
    a = assessments[pid]
    delivery = a.delivery_risk.severity.value if a.delivery_risk else "N/A (unmapped)"
    financial = a.financial_risk.severity.value if a.financial_risk else "N/A (unmapped)"
    print(f"{pid}: delivery={delivery:16s} financial={financial:16s} -> combined={a.combined_rag.value}")

PROJECT-10006: delivery=MEDIUM           financial=N/A (unmapped)   -> combined=AMBER
PROJECT-10007: delivery=N/A (unmapped)   financial=LOW              -> combined=AMBER


## Validation checks

- [x] All 7 projects assessed without raising
- [x] All 5 fully-mapped projects computed RED — matching Phase 1's independent, hand-computed prediction exactly
- [x] Every HIGH-severity Risk carries non-empty evidence
- [x] Both mapping-gap projects cap at AMBER, including Helios despite a LOW financial risk underneath
- [x] `reason_codes` on each assessment is a deduplicated, sorted union of both sides

## Testing

`tests/test_delivery_metrics.py` (23 tests), `tests/test_risk_engine.py` (11 tests) — `pytest tests/test_delivery_metrics.py tests/test_risk_engine.py -v`.

## Next step

Phase 7: Mem0 historical memory — persisting this week's `ProjectSnapshot` per project so `sprint_count_blocked` and week-over-week trend classification (currently `BASELINE` for everything, since no history exists) can finally be computed for real.